# 抽出品質の検証 — v0.3 (3層抽出戦略)

v0.3 の `extractors.py` 改修 (要素ID マッチ層・項目名マッチ層・LLM フォールバック層) が、旧来の text_fallback 中心の抽出と比べてどの程度品質を改善したかを定量的に検証する。

## チェック観点
1. **抽出経路 (`matched_by`) の分布** — `element_id_match` が支配的になっていれば成功
2. **F=M同値率** — 旧バグでは 132/140 (94%) だったものが大幅減しているか
3. **指標カバー率** — `gender_wage_gap` の取得率が 64% から 95%+ に改善したか
4. **次元の分布** — `(scope, worker_type)` で正しく層別化されているか

事前に `edinet backfill --from 2024-01-01 --to 2024-12-31 --retry-failed` を実行して既存データを新ロジックで上書きしてから本ノートを実行する。

In [ ]:
import duckdb
import pandas as pd

DUCKDB_PATH = '../artifacts/analytics/edinet_analytics.duckdb'
con = duckdb.connect(DUCKDB_PATH, read_only=True)
con.execute('SHOW TABLES').fetchdf()

## 1. matched_by 分布 — Layer 1/2/3 の貢献度

In [ ]:
df_matched = con.execute("""
    SELECT metric_name, matched_by, COUNT(*) AS n
    FROM analytics.metric_evidence
    WHERE metric_name IN ('female_manager_ratio','male_childcare_leave_ratio','gender_wage_gap')
    GROUP BY metric_name, matched_by
    ORDER BY metric_name, n DESC
""").fetchdf()
df_matched.pivot(index='metric_name', columns='matched_by', values='n').fillna(0).astype(int)

**期待値**: 2024年度以降の書類では `element_id_match` が 80%+ を占めるはず。
`text_fallback` が大幅減 / `llm_fallback` は LLM_FALLBACK_ENABLED=true 時のみ出現。

## 2. F=M同値バグの根絶確認

In [ ]:
df_bug = con.execute("""
    SELECT 
        SUM(CASE WHEN female_manager_ratio = male_childcare_leave_ratio 
                  AND female_manager_ratio IS NOT NULL THEN 1 ELSE 0 END) AS f_eq_m,
        SUM(CASE WHEN female_manager_ratio = gender_wage_gap 
                  AND female_manager_ratio IS NOT NULL THEN 1 ELSE 0 END) AS f_eq_w,
        COUNT(*) AS total_records
    FROM analytics.company_year_metrics
    WHERE scope = 'reporting_company' AND worker_type = 'all'
      AND female_manager_ratio IS NOT NULL
""").fetchdf()
df_bug

**目標**: f_eq_m / total_records < 5% (旧 94%)。偶然の一致のみが残る想定。

## 3. 指標カバー率 (デフォルト次元)

In [ ]:
con.execute("""
    SELECT 
        fiscal_year,
        COUNT(*) AS total,
        ROUND(100.0 * COUNT(female_manager_ratio) / COUNT(*), 1) AS female_mgr_pct,
        ROUND(100.0 * COUNT(male_childcare_leave_ratio) / COUNT(*), 1) AS childcare_pct,
        ROUND(100.0 * COUNT(gender_wage_gap) / COUNT(*), 1) AS wage_gap_pct
    FROM analytics.company_year_metrics
    WHERE scope='reporting_company' AND worker_type='all' AND status='processed'
    GROUP BY fiscal_year
    ORDER BY fiscal_year
""").fetchdf()

**目標**: `wage_gap_pct` が 64% → 95%+ に改善。

## 4. 次元の分布 — scope × worker_type

In [ ]:
con.execute("""
    SELECT scope, worker_type, COUNT(*) AS records,
           COUNT(female_manager_ratio) AS f_count,
           COUNT(male_childcare_leave_ratio) AS m_count,
           COUNT(gender_wage_gap) AS w_count
    FROM analytics.company_year_metrics
    WHERE status = 'processed'
    GROUP BY scope, worker_type
    ORDER BY scope, worker_type
""").fetchdf()

## 5. サンプル監査 — 5社抜き取りで evidence を見る

In [ ]:
con.execute("""
    SELECT company_name, fiscal_year, metric_name, matched_by, scope, worker_type, raw_value
    FROM analytics.metric_evidence
    WHERE metric_name IN ('female_manager_ratio','male_childcare_leave_ratio','gender_wage_gap')
      AND fiscal_year >= 2024
    ORDER BY company_name, fiscal_year, metric_name
    LIMIT 30
""").fetchdf()